> 在线讲义：[第 12 章 · 价值迭代与 GPI：评估只算一轮就改进](https://wikieden.github.io/rl-from-scratch/lessons/12-value-iteration.html) · 本书《从零学强化学习》

# 第 12 章 · 价值迭代与 GPI：评估只算一轮就改进

本章回答：能不能把"评估"和"改进"合成一步？价值迭代和第 9 章的最优方程是什么关系？为什么说几乎所有 RL 算法都是"广义策略迭代"？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/12-value-iteration.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 4×4 价值迭代 | `value_iteration(P, 1.0)` | 4 轮，0、−1、−2、−3 … |
| 读出策略 | `greedy_policy(P, v, 1.0)` | 箭头指向最近终点 |
| 5×5 价值迭代 | `value_iteration(G, 0.9)` | 154 轮，$v_*(A) = 24.4$ |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `q_from_v`**（来自 `ch08_bellman.py`）：q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。

In [ ]:
def q_from_v(P, v, gamma, s, a):
    """q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。"""
    return sum(p * (r + (0 if done else gamma * v[ns])) for p, ns, r, done in P[s][a])

**函数 `greedy_policy`**（来自 `ch11_policy_iteration.py`）：每个状态在 q 最大的动作之间平分概率（并列时都留着，方便看）。

In [ ]:
def greedy_policy(P, v, gamma, eps=1e-9):
    """每个状态在 q 最大的动作之间平分概率（并列时都留着，方便看）。"""
    pi = {}
    for s in P:
        qs = {a: q_from_v(P, v, gamma, s, a) for a in P[s]}
        best = max(qs.values())
        top = [a for a, q in qs.items() if q > best - eps]
        pi[s] = {a: (1 / len(top) if a in top else 0.0) for a in P[s]}
    return pi

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `gridworld_4x4`**（来自 `rl_envs.py`）：S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。

In [ ]:
def gridworld_4x4():
    """S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。"""
    n = 4
    P = {}
    for s in range(n * n):
        r, c = divmod(s, n)
        P[s] = {}
        for a, (dr, dc) in enumerate(MOVES):
            if s in (0, 15):
                P[s][a] = [(1.0, s, 0.0, True)]
                continue
            nr, nc = r + dr, c + dc
            ns = nr * n + nc if (0 <= nr < n and 0 <= nc < n) else s
            P[s][a] = [(1.0, ns, -1.0, ns in (0, 15))]
    return P

**函数 `gridworld_5x5`**（来自 `rl_envs.py`）：S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，

In [ ]:
def gridworld_5x5():
    """S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，
    撞墙原地不动 −1，其余 0。连续任务，不结束。状态编号 = 行 × 5 + 列。"""
    n = 5
    P = {}
    for r in range(n):
        for c in range(n):
            s = r * n + c
            P[s] = {}
            for a, (dr, dc) in enumerate(MOVES):
                if (r, c) == (0, 1):
                    P[s][a] = [(1.0, 4 * n + 1, 10.0, False)]
                elif (r, c) == (0, 3):
                    P[s][a] = [(1.0, 2 * n + 3, 5.0, False)]
                else:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n:
                        P[s][a] = [(1.0, nr * n + nc, 0.0, False)]
                    else:
                        P[s][a] = [(1.0, s, -1.0, False)]
    return P

**常量 `MOVE_NAMES`**（来自 `rl_envs.py`）

In [ ]:
MOVE_NAMES = ["↑", "↓", "→", "←"]

**函数 `show_policy`**（来自 `ch11_policy_iteration.py`）：下面要用到的小工具。

In [ ]:
def show_policy(pi, n):
    rows = []
    for r in range(n):
        rows.append(" ".join("".join(MOVE_NAMES[a] for a, p in pi[r * n + c].items() if p > 0) or "·"
                             for c in range(n)))
    return "\n".join(rows)

**函数 `value_iteration`**（来自 `ch12_value_iteration.py`）：下面要用到的小工具。

In [ ]:
def value_iteration(P, gamma, theta=1e-6):
    v, sweeps = np.zeros(len(P)), 0
    while True:
        new = np.array([max(q_from_v(P, v, gamma, s, a) for a in P[s]) for s in P])
        sweeps += 1
        if np.max(np.abs(new - v)) < theta:
            return new, sweeps
        v = new

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P = gridworld_4x4()

In [ ]:
v, n = value_iteration(P, 1.0)

In [ ]:
print(f"4×4 价值迭代 {n} 轮：\n{np.round(v.reshape(4, 4), 1)}")

In [ ]:
print(show_policy(greedy_policy(P, v, 1.0), 4))

In [ ]:
G = gridworld_5x5()

In [ ]:
vg, ng = value_iteration(G, 0.9)

In [ ]:
print(f"5×5 价值迭代 {ng} 轮，v*(A) = {vg[1]:.1f}")

## 3. 代码里才看得到的

和第 10 章 `sweep` 比，只是把 `sum(pa * ...)` 换成了 `max(...)`。5×5 网格要 154 轮是因为 γ = 0.9：误差每轮只缩成 0.9 倍，要从 24 缩到 $10^{-6}$，大约要 $\log(24 \times 10^6) / \log(1/0.9) \approx 160$ 轮。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 同一个 5×5 网格做价值迭代：γ = 0.5 只要 25 轮，γ = 0.99 要 1605 轮，算出的 $v_*(A)$ 和第 11 章一样（10.323、204.04）。价值迭代的轮数随 γ 涨得很快，策略迭代不会。
2. 把 `theta` 调大到 `1e-2`，看轮数和 $v_*(A)$ 各变了多少。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 评估一轮就改进，合成一个 max；
> 价值迭代套最优方程，策略评估套期望方程；
> 不必扫遍，挑着更新也收敛；
> 评估改进你推我拉，所有方法都是 GPI；DP 要知道 p。